In [3]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt
from langgraph.prebuilt import create_react_agent
from dotenv import load_dotenv
load_dotenv()

# An example of a sensitive tool that requires human review / approval
def book_hotel(hotel_name: str):
    """Book a hotel"""
    # highlight-next-line
    response = interrupt(  # (1)!
        f"Trying to call `book_hotel` with args {{'hotel_name': {hotel_name}}}. "
        "Please approve or suggest edits."
    )
    if response["type"] == "accept":
        pass
    elif response["type"] == "edit":
        hotel_name = response["args"]["hotel_name"]
    else:
        raise ValueError(f"Unknown response type: {response['type']}")
    return f"Successfully booked a stay at {hotel_name}."

# highlight-next-line
checkpointer = InMemorySaver() # (2)!

agent = create_react_agent(
    model="gpt-4.1",
    tools=[book_hotel],
    # highlight-next-line
    checkpointer=checkpointer, # (3)!
)

In [4]:
config = {
   "configurable": {
      # highlight-next-line
      "thread_id": "1"
   }
}

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "book a stay at McKittrick hotel"}]},
    # highlight-next-line
    config
):
    print(chunk)
    print("\n")

{'agent': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_7HMYdPPy8y4SeLddOXZaSOy4', 'function': {'arguments': '{"hotel_name":"McKittrick hotel"}', 'name': 'book_hotel'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 52, 'total_tokens': 72, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_ccc6ec921d', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--c9dab49a-1f48-4f76-ac9c-c2f0630125db-0', tool_calls=[{'name': 'book_hotel', 'args': {'hotel_name': 'McKittrick hotel'}, 'id': 'call_7HMYdPPy8y4SeLddOXZaSOy4', 'type': 'tool_call'}], usage_metadata={'input_tokens': 52, 'output_tokens': 20, 'total_tokens': 72, 'input_token_details': {'audio': 0, 'cache_read': 0}, 

In [5]:
from langgraph.types import Command

for chunk in agent.stream(
    # highlight-next-line
    Command(resume={"type": "accept"}),  # (1)!
    # Command(resume={"type": "edit", "args": {"hotel_name": "McKittrick Hotel"}}),
    config
):
    print(chunk)
    print("\n")

{'tools': {'messages': [ToolMessage(content='Successfully booked a stay at McKittrick hotel.', name='book_hotel', id='b90e41f3-1f7d-4c9d-b822-b8ee41ad98a1', tool_call_id='call_7HMYdPPy8y4SeLddOXZaSOy4')]}}


{'agent': {'messages': [AIMessage(content='Your stay at the McKittrick Hotel has been successfully booked. If you need further details or wish to modify your booking, please let me know!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 92, 'total_tokens': 124, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_ccc6ec921d', 'finish_reason': 'stop', 'logprobs': None}, id='run--425e5ec8-ba8a-48b1-9312-e0e727a8122c-0', usage_metadata={'input_tokens': 92, 'output_tokens': 32, 'total_tokens': 124, 'input_toke